## Imports

In [ ]:
import joblib
import torch

import time
import numpy as np
import pandas as pd

from transformers import AutoModelForSequenceClassification, AutoTokenizer

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Loading the test split

In [ ]:
test_df = joblib.load('drive/MyDrive/nlp/test_df.pkl')
test_df = test_df['text']

In [ ]:
test_df.head()

,text
0,"We are in the Yarra valley, we have a pair of ..."
1,Been here twice and had a great time. Location...
2,"Was it a ""core"" or ""non-core"" commitment?\nThe..."
3,Anyone proposing to run as a candidate for ele...
4,Yeah well out of this doctor and what he actua...


## Loading the models

In [ ]:
lr_vectorizer = joblib.load("drive/MyDrive/nlp/sent_lr_vectorizer.joblib")

In [ ]:
lr_model = joblib.load("drive/MyDrive/nlp/sentiment_seed42.joblib")

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_path = "/content/drive/MyDrive/nlp/roberta_sentiment_seed42"

roberta_model = AutoModelForSequenceClassification.from_pretrained(model_path).to(device)
tokenizer = AutoTokenizer.from_pretrained(model_path)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

### Converting the test df using the TF-IDF vectorizer

In [ ]:
test_lr = lr_vectorizer.transform(test_df)

## Inference functions

In [ ]:
def lr_predict(model, X):
    return model.predict(X)

In [ ]:
def roberta_predict(model, tokenizer, texts, device="cpu"):
    model.eval()

    inputs = tokenizer(
        texts,
        padding=True,
        truncation=True,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        outputs = model(**inputs)
        preds = torch.argmax(outputs.logits, dim=1)

    return preds.cpu().numpy()

## Helper functions

In [ ]:
def measure_time(fn, inputs, n_runs=20):
    times = []

    for _ in range(n_runs):
        start = time.time()
        fn(inputs)
        end = time.time()
        times.append((end - start) * 1000)

    return np.mean(times), np.std(times)

In [ ]:
def stress_test(fn, data, n_requests=1000):
    start = time.time()

    for _ in range(n_requests):
        fn(data[:1])

    end = time.time()

    total = end - start
    return total, n_requests / total

In [ ]:
def max_batch_capacity_test(fn, data, batch_sizes=None):
    if batch_sizes is None:
        batch_sizes = [1, 8, 32, 64, 128, 256, 512, 1024]

    results = []

    for b in batch_sizes:
        try:
            start = time.time()
            fn(data[:b])
            end = time.time()

            results.append({
                "batch_size": b,
                "status": "OK",
                "time_ms": (end - start) * 1000
            })

            print(f"Batch {b}: OK")

        except Exception as e:
            results.append({
                "batch_size": b,
                "status": "FAILED",
                "error": str(e)
            })

            print(f"Batch {b}: FAILED → {e}")
            break

    return results

In [ ]:
def stress_test(fn, data, n_requests=500):
    start = time.time()

    for _ in range(n_requests):
        fn(data[:1])

    end = time.time()

    total = end - start
    return total, n_requests / total

# CPU test

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cpu')

## Single input latency

In [ ]:
lr_cpu_single = measure_time(lambda x: lr_predict(lr_model, x), test_lr[:1])

In [ ]:
rb_cpu_single = measure_time(lambda x: roberta_predict(roberta_model, tokenizer, x, device), test_df[:1].to_list())

In [ ]:
lr_cpu_single, rb_cpu_single

((np.float64(0.23664236068725586), np.float64(0.3624965227420981)),
 (np.float64(384.1027021408081), np.float64(1278.0049660673878)))

## Batch scaling

In [ ]:
batch_sizes = [1, 8, 32, 128]

lr_cpu_batch_results = []
rb_cpu_batch_results = []

for b in batch_sizes:
    lr_time, _ = measure_time(lambda x: lr_predict(lr_model, x), test_lr[:b])
    rb_time, _ = measure_time(lambda x: roberta_predict(roberta_model, tokenizer, x, device), test_df[:b].to_list())

    lr_cpu_batch_results.append([b,lr_time])
    rb_cpu_batch_results.append([b, rb_time])

In [ ]:
lr_cpu_batch_results, rb_cpu_batch_results

([[1, np.float64(0.1736283302307129)],
  [8, np.float64(0.14551877975463867)],
  [32, np.float64(0.14269351959228516)],
  [128, np.float64(0.1627802848815918)]],
 [[1, np.float64(70.78356742858887)],
  [8, np.float64(2637.2286677360535)],
  [32, np.float64(9801.822245121002)],
  [128, np.float64(35898.063576221466)]])

## Stress test

In [ ]:
lr_cpu_stress = stress_test(lambda x: lr_predict(lr_model, x), test_lr)
rb_cpu_stress = stress_test(lambda x: roberta_predict(roberta_model, tokenizer, x, device), test_df.to_list())

In [ ]:
lr_cpu_stress, rb_cpu_stress

((0.13627982139587402, 3668.921744089825),
 (38.05811357498169, 13.137803034165247))

## Long text test

In [ ]:
long_text = ["This movie was great. " * 200]
long_text_lr = lr_vectorizer.transform(long_text)

In [ ]:
lr_cpu_long = measure_time(lambda x: lr_predict(lr_model, x), long_text_lr)
rb_cpu_long = measure_time(lambda x: roberta_predict(roberta_model, tokenizer, x, device), long_text)

In [ ]:
lr_cpu_long, rb_cpu_long

((np.float64(0.12313127517700195), np.float64(0.047290640691229145)),
 (np.float64(160.39972305297852), np.float64(38.46478522170797)))

## Batch Size capacity test

In [ ]:
lr_cpu_capacity = max_batch_capacity_test(
    lambda x: lr_predict(lr_model, x),
    test_lr
)

rb_cpu_capacity = max_batch_capacity_test(
    lambda x: roberta_predict(roberta_model, tokenizer, x, device),
    test_df.tolist()
)

Batch 1: OK
Batch 8: OK
Batch 32: OK
Batch 64: OK
Batch 128: OK
Batch 256: OK
Batch 512: OK
Batch 1024: OK
Batch 1: OK
Batch 8: OK
Batch 32: OK
Batch 64: OK
Batch 128: OK
Batch 256: OK
Batch 512: OK
Batch 1024: OK


In [ ]:
lr_cpu_capacity, rb_cpu_capacity

([{'batch_size': 1, 'status': 'OK', 'time_ms': 0.5862712860107422},
  {'batch_size': 8, 'status': 'OK', 'time_ms': 0.4019737243652344},
  {'batch_size': 32, 'status': 'OK', 'time_ms': 0.32329559326171875},
  {'batch_size': 64, 'status': 'OK', 'time_ms': 0.30684471130371094},
  {'batch_size': 128, 'status': 'OK', 'time_ms': 0.9088516235351562},
  {'batch_size': 256, 'status': 'OK', 'time_ms': 0.5142688751220703},
  {'batch_size': 512, 'status': 'OK', 'time_ms': 0.8420944213867188},
  {'batch_size': 1024, 'status': 'OK', 'time_ms': 0.9579658508300781}],
 [{'batch_size': 1, 'status': 'OK', 'time_ms': 83.7252140045166},
  {'batch_size': 8, 'status': 'OK', 'time_ms': 1964.0765190124512},
  {'batch_size': 32, 'status': 'OK', 'time_ms': 9096.230506896973},
  {'batch_size': 64, 'status': 'OK', 'time_ms': 17704.147338867188},
  {'batch_size': 128, 'status': 'OK', 'time_ms': 36302.780628204346},
  {'batch_size': 256, 'status': 'OK', 'time_ms': 91734.30252075195},
  {'batch_size': 512, 'status': 

# GPU test

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

## Single input latency

In [ ]:
lr_gpu_single = measure_time(lambda x: lr_predict(lr_model, x), test_lr[:1])

In [ ]:
rb_gpu_single = measure_time(lambda x: roberta_predict(roberta_model, tokenizer, x, device), test_df[:1].to_list())

In [ ]:
lr_gpu_single, rb_gpu_single

((np.float64(0.6371378898620605), np.float64(0.929906864352886)),
 (np.float64(14.997124671936035), np.float64(4.9301500629529755)))

## Batch scaling

In [ ]:
batch_sizes = [1, 8, 32, 128]

lr_gpu_batch_results = []
rb_gpu_batch_results = []

for b in batch_sizes:
    lr_time, _ = measure_time(lambda x: lr_predict(lr_model, x), test_lr[:b])
    rb_time, _ = measure_time(lambda x: roberta_predict(roberta_model, tokenizer, x, device), test_df[:b].to_list())

    lr_gpu_batch_results.append([b,lr_time])
    rb_gpu_batch_results.append([b, rb_time])

In [ ]:
lr_gpu_batch_results, rb_gpu_batch_results

([[1, np.float64(0.21638870239257812)],
  [8, np.float64(0.1265883445739746)],
  [32, np.float64(0.13369321823120117)],
  [128, np.float64(0.1534104347229004)]],
 [[1, np.float64(11.141490936279297)],
  [8, np.float64(146.39595746994019)],
  [32, np.float64(491.84913635253906)],
  [128, np.float64(2072.7869749069214)]])

## Stress test

In [ ]:
lr_gpu_stress = stress_test(lambda x: lr_predict(lr_model, x), test_lr)
rb_gpu_stress = stress_test(lambda x: roberta_predict(roberta_model, tokenizer, x, device), test_df.to_list())

In [ ]:
lr_gpu_stress, rb_gpu_stress

((0.15767693519592285, 3171.040833453039),
 (4.353339672088623, 114.85435037512535))

## Long text test

In [ ]:
long_text = ["This movie was great. " * 200]
long_text_lr = lr_vectorizer.transform(long_text)

In [ ]:
lr_gpu_long = measure_time(lambda x: lr_predict(lr_model, x), long_text_lr)
rb_gpu_long = measure_time(lambda x: roberta_predict(roberta_model, tokenizer, x, device), long_text)

In [ ]:
lr_gpu_long, rb_gpu_long

((np.float64(0.24074316024780273), np.float64(0.08358623096952329)),
 (np.float64(39.64345455169678), np.float64(4.423495282201753)))

## Batch Size capacity test

In [ ]:
lr_gpu_capacity = max_batch_capacity_test(
    lambda x: lr_predict(lr_model, x),
    test_lr
)

rb_gpu_capacity = max_batch_capacity_test(
    lambda x: roberta_predict(roberta_model, tokenizer, x, device),
    test_df.tolist()
)

Batch 1: OK
Batch 8: OK
Batch 32: OK
Batch 64: OK
Batch 128: OK
Batch 256: OK
Batch 512: OK
Batch 1024: OK
Batch 1: OK
Batch 8: OK
Batch 32: OK
Batch 64: OK
Batch 128: OK
Batch 256: OK
Batch 512: OK
Batch 1024: FAILED → CUDA out of memory. Tried to allocate 4.70 GiB. GPU 0 has a total capacity of 14.56 GiB of which 3.42 GiB is free. Including non-PyTorch memory, this process has 11.14 GiB memory in use. Of the allocated memory 7.68 GiB is allocated by PyTorch, and 3.33 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)


In [ ]:
lr_gpu_capacity, rb_gpu_capacity

([{'batch_size': 1, 'status': 'OK', 'time_ms': 1.5039443969726562},
  {'batch_size': 8, 'status': 'OK', 'time_ms': 0.9107589721679688},
  {'batch_size': 32, 'status': 'OK', 'time_ms': 0.3428459167480469},
  {'batch_size': 64, 'status': 'OK', 'time_ms': 0.7560253143310547},
  {'batch_size': 128, 'status': 'OK', 'time_ms': 0.42366981506347656},
  {'batch_size': 256, 'status': 'OK', 'time_ms': 0.4189014434814453},
  {'batch_size': 512, 'status': 'OK', 'time_ms': 0.4608631134033203},
  {'batch_size': 1024, 'status': 'OK', 'time_ms': 1.562356948852539}],
 [{'batch_size': 1, 'status': 'OK', 'time_ms': 14.48512077331543},
  {'batch_size': 8, 'status': 'OK', 'time_ms': 160.12907028198242},
  {'batch_size': 32, 'status': 'OK', 'time_ms': 613.7516498565674},
  {'batch_size': 64, 'status': 'OK', 'time_ms': 1317.8834915161133},
  {'batch_size': 128, 'status': 'OK', 'time_ms': 2406.299591064453},
  {'batch_size': 256, 'status': 'OK', 'time_ms': 6655.935764312744},
  {'batch_size': 512, 'status': 'O

# Results

In [ ]:
import pandas as pd
import numpy as np

data = [
    # ---------- CPU ----------
    ["CPU", "LR", "Single", None, 0.2366, "OK"],
    ["CPU", "RoBERTa", "Single", None, 384.1027, "OK"],

    ["CPU", "LR", "Batch", 1, 0.1736, "OK"],
    ["CPU", "LR", "Batch", 8, 0.1455, "OK"],
    ["CPU", "LR", "Batch", 32, 0.1427, "OK"],
    ["CPU", "LR", "Batch", 128, 0.1628, "OK"],

    ["CPU", "RoBERTa", "Batch", 1, 70.7836, "OK"],
    ["CPU", "RoBERTa", "Batch", 8, 2637.2287, "OK"],
    ["CPU", "RoBERTa", "Batch", 32, 9801.8222, "OK"],
    ["CPU", "RoBERTa", "Batch", 128, 35898.0636, "OK"],

    ["CPU", "LR", "Stress", None, 0.1363, "OK"],
    ["CPU", "RoBERTa", "Stress", None, 38.0581, "OK"],

    ["CPU", "LR", "Long", None, 0.1231, "OK"],
    ["CPU", "RoBERTa", "Long", None, 160.3997, "OK"],

    # ---------- GPU ----------
    ["GPU", "LR", "Single", None, 0.6371, "OK"],
    ["GPU", "RoBERTa", "Single", None, 14.9971, "OK"],

    ["GPU", "LR", "Batch", 1, 0.2164, "OK"],
    ["GPU", "LR", "Batch", 8, 0.1266, "OK"],
    ["GPU", "LR", "Batch", 32, 0.1337, "OK"],
    ["GPU", "LR", "Batch", 128, 0.1534, "OK"],

    ["GPU", "RoBERTa", "Batch", 1, 11.1415, "OK"],
    ["GPU", "RoBERTa", "Batch", 8, 146.3960, "OK"],
    ["GPU", "RoBERTa", "Batch", 32, 491.8491, "OK"],
    ["GPU", "RoBERTa", "Batch", 128, 2072.7870, "OK"],

    ["GPU", "LR", "Stress", None, 0.1577, "OK"],
    ["GPU", "RoBERTa", "Stress", None, 4.3533, "OK"],

    ["GPU", "LR", "Long", None, 0.2407, "OK"],
    ["GPU", "RoBERTa", "Long", None, 39.6435, "OK"],

    # ---------- Capacity ----------
    ["CPU", "LR", "Capacity", 1024, 0.9580, "OK"],
    ["CPU", "RoBERTa", "Capacity", 1024, 387657.9735, "OK"],

    ["GPU", "LR", "Capacity", 1024, 1.5624, "OK"],
    ["GPU", "RoBERTa", "Capacity", 1024, np.nan, "FAILED"],
]

df_all = pd.DataFrame(
    data,
    columns=["Device", "Model", "TestType", "BatchSize", "Time_ms", "Status"]
)

df_all

,Device,Model,TestType,BatchSize,Time_ms,Status
0,CPU,LR,Single,NaN,0.2366,OK
1,CPU,RoBERTa,Single,NaN,384.1027,OK
2,CPU,LR,Batch,1.0,0.1736,OK
3,CPU,LR,Batch,8.0,0.1455,OK
4,CPU,LR,Batch,32.0,0.1427,OK
5,CPU,LR,Batch,128.0,0.1628,OK
6,CPU,RoBERTa,Batch,1.0,70.7836,OK
7,CPU,RoBERTa,Batch,8.0,2637.2287,OK
8,CPU,RoBERTa,Batch,32.0,9801.8222,OK
9,CPU,RoBERTa,Batch,128.0,35898.0636,OK
